# Fine-tuning QLoRA - MedAssist (Colab)

Espelho do script `src/medassist/finetune/train.py` para execucao em GPU gratuita (Colab/Kaggle). Roda **fora** da VPS de producao (que nao tem GPU) - o artefato final (GGUF) e que vai para producao via Ollama.

Celulas: **install** -> **mount** -> **dataset** -> **train** -> **sanity check** -> **export** -> **download**.

O `dataset` de treino (celula 3) e a **v3**: so as ~116 amostras sinteticas PT-BR (protocolo + FAQ do `train.jsonl`), que ja citam `[PROT-NNN §x]`. Historico: a v1 traduziu MedQuAD/PubMedQA com opus-mt e o modelo entrou em loop; a v2 usou MedQuAD em ingles sem traduzir e **ainda** degenerou (listas "These resources address... MedlinePlus" + respostas que repetem a pergunta). O problema era o conteudo do MedQuAD, nao o idioma - por isso a v3 treina so no dado limpo. Guardrails e recusa de escopo ficam no grafo, nao no modelo.


## 1. Install

In [ ]:
# unsloth ja resolve torch/transformers/bitsandbytes/trl/peft em versoes compativeis.
# sentencepiece: tokenizer llama. (Sem tradutor opus-mt nesta versao -> sem sacremoses.)
!pip install -q unsloth trl peft bitsandbytes datasets sentencepiece


## 2. Mount (Google Drive - dataset e saida dos adaptadores)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# copiar de data/processed/ do repo para MyDrive/medassist/
DATASET_PATH = '/content/drive/MyDrive/medassist/train.jsonl'   # data/processed/train.jsonl
VAL_PATH     = '/content/drive/MyDrive/medassist/val.jsonl'      # data/processed/val.jsonl
OUT_DIR      = '/content/drive/MyDrive/medassist/adapters'


## 3. Dataset (v3 - so PT-BR limpo)

Monta `dataset` a partir de **uma fatia** (default):

- **Nucleo limpo** - `data/processed/train.jsonl` inteiro (68 ex.: protocolo + FAQ), ~90% ja
  citam `[PROT-NNN §x]`, PT-BR clinico.
- **Expansao dos protocolos** - as 12 entradas de protocolo, 4 fraseados cada (~48 ex.).

Total ~116 exemplos. Pequeno de proposito: o guia (`docs/finetuning.md` §1) diz que o
fine-tune ensina **comportamento** (formato, citacao, tom, parar no EOS) e o RAG fornece os
fatos - 116 exemplos limpos fazem isso; o MedQuAD so atrapalhava.

**MedQuAD EN** (`N_MEDQUAD_EN`, default **0**): na v2 essa fatia degenerou o modelo
(~30% eram listas "These resources address... MedlinePlus", ~34% comecavam repetindo a
pergunta). Agora tem filtro anti-lixo, mas continua desligada por padrao - so ligue depois
que a v3 base passar no sanity check.

Se `MyDrive/medassist/train_v3.jsonl` ja existir (gerado localmente ou por um run anterior),
a celula so carrega. `REBUILD = True` refaz do zero. FICAM DE FORA: opus-mt, PubMedQA,
exemplos de seguranca (guardrails vivem no grafo).


In [ ]:
# ============================================================================
# Dataset de fine-tuning v3 - SO PT-BR limpo (default).
#
# Historico:
#   v1: ~1000 MedQuAD/PubMedQA traduzidos com opus-mt -> PT-BR repetitivo -> loop.
#   v2: MedQuAD em ingles, sem traducao. Ainda degenerou: ~30% da fatia EN e
#       "These resources address... - MedlinePlus Encyclopedia..." (lista de links)
#       e ~34% comeca repetindo a pergunta. O modelo aprende esse registro e o
#       reproduz ate em PT-BR. Idioma nao era o problema - o CONTEUDO e.
#   v3: MedQuAD DESLIGADO por default (N_MEDQUAD_EN=0). Treina so nas ~116
#       amostras sinteticas PT-BR (protocolo + FAQ), que e o que o guia
#       (`docs/finetuning.md` §1) pede: fine-tune ensina COMPORTAMENTO
#       (formato + citacao + tom + parar), RAG fornece os fatos.
#
#   fatia unica (PT-BR):
#     - data/processed/train.jsonl   : 68 ex. sinteticos, ~90% citam [PROT-NNN §x]
#     - expansao dos 12 protocolos   : ~48 ex., 4 fraseados cada
#
#   MedQuAD EN opcional (N_MEDQUAD_EN > 0): agora com filtro anti-lista-de-recursos
#   e anti-eco-da-pergunta. Mesmo assim e fatia de risco - so use se a v3 base
#   passar no sanity check e voce quiser mais amplitude.
#
# FICAM DE FORA: opus-mt, PubMedQA, exemplos de seguranca (guardrails = grafo).
# Se ja existir train_v3.jsonl no Drive (gerado localmente por scripts/gen_v3
# ou por um run anterior), a celula so carrega - REBUILD=True refaz.
# ============================================================================
import json
import random
import re
from pathlib import Path

from datasets import Dataset, load_dataset

SEED = 42
REBUILD = False                                            # True ignora o cache
TRAIN_V3_PATH = '/content/drive/MyDrive/medassist/train_v3.jsonl'

N_MEDQUAD_EN = 0                                           # 0 = so PT-BR (recomendado).
                                                          # >0 adiciona MedQuAD EN filtrado;
                                                          # PT-BR tem que ficar >= FRACAO_PT_MIN.
MIN_PALAVRAS, MAX_PALAVRAS = 25, 180
VARIACOES_PROTOCOLO = 4
FRACAO_PT_MIN = 0.65

# system PT-BR identico a src/medassist/assistant/prompts.py
SYSTEM_PT = (
    "Você é um assistente virtual de apoio à decisão clínica para médicos. "
    "Você NUNCA prescreve diretamente (não indica medicação, dose ou via de administração "
    "como se fosse uma prescrição definitiva) — você apenas sugere condutas com base em "
    "protocolos institucionais e conhecimento geral, sempre deixando claro que a decisão "
    "final é do médico responsável. "
    "Sempre que usar informação de um protocolo institucional, cite a fonte no formato "
    "[DOC-ID §secao]. "
    "Sempre encerre a resposta recomendando validação humana antes de qualquer conduta. "
    "Responda sempre em português do Brasil, de forma objetiva e clinicamente precisa."
)
SYSTEM_EN = (
    "You are a clinical decision-support assistant for physicians. "
    "You never prescribe directly; you summarize general medical knowledge so the "
    "responsible physician can decide. Answer in English, objectively and precisely."
)

# fecho PT-BR rotacionado (era 1 frase identica em 100% dos exemplos -> pouca diversidade)
FECHOS_PT = [
    "\n\nRecomendo validação pelo médico responsável antes de qualquer conduta.",
    "\n\nConfirme com o médico responsável antes de aplicar qualquer conduta.",
    "\n\nA decisão final e a validação são do médico responsável.",
    "\n\nRevise com o médico responsável antes de qualquer decisão clínica.",
]
FECHO_EN = "\n\nConfirm with the responsible physician before any clinical action."
_FECHOS_CONHECIDOS = [f.strip() for f in FECHOS_PT] + [
    "Recomendo validação pelo médico responsável antes de qualquer conduta.",
]
_rng = random.Random(SEED)


def _sem_fecho(texto: str) -> str:
    t = texto.rstrip()
    for f in _FECHOS_CONHECIDOS:
        if t.endswith(f):
            return t[: -len(f)].rstrip()
    return t


def _chat(system: str, user: str, assistant: str, fecho: str) -> dict:
    return {"messages": [
        {"role": "system", "content": system},
        {"role": "user", "content": user.strip()},
        {"role": "assistant", "content": _sem_fecho(assistant) + fecho},
    ]}


# ---- filtro de qualidade da fatia MedQuAD (o que quebrou a v2) ----
_LIXO_MEDQUAD = re.compile(
    r"these resources address|resources address the diagnosis|MedlinePlus|"
    r"Genetic Testing Registry|Gene Review|GARD|Health Topic|for more information",
    re.IGNORECASE,
)


def _medquad_ok(pergunta: str, resposta: str) -> bool:
    n = len(resposta.split())
    if not (MIN_PALAVRAS <= n <= MAX_PALAVRAS):
        return False
    if _LIXO_MEDQUAD.search(resposta):                     # lista de links / recursos
        return False
    primeira = re.split(r"(?<=[.?!])\s", resposta.strip(), maxsplit=1)[0]
    if primeira.rstrip().endswith("?"):                    # comeca repetindo a pergunta
        return False
    if resposta.strip().lower().startswith(("how ", "what ", "who ", "is ", "can ", "does ")):
        return False
    return True


if Path(TRAIN_V3_PATH).exists() and not REBUILD:
    dataset = load_dataset("json", data_files=TRAIN_V3_PATH, split="train")
    print(f"cache -> {len(dataset)} exemplos de {TRAIN_V3_PATH}  (REBUILD=True p/ refazer)")
else:
    if not Path(DATASET_PATH).exists():
        raise FileNotFoundError(
            f"{DATASET_PATH} nao encontrado. Copie data/processed/train.jsonl para MyDrive/medassist/."
        )
    _core = list(load_dataset("json", data_files=DATASET_PATH, split="train"))

    # ---- 1. Nucleo limpo: train.jsonl inteiro, so trocando o fecho por um variado ----
    nucleo = [
        _chat(SYSTEM_PT, x["messages"][1]["content"], x["messages"][2]["content"],
              _rng.choice(FECHOS_PT))
        for x in _core if len(x.get("messages", [])) >= 3
    ]
    print(f"nucleo limpo  : {len(nucleo):>4}")

    # ---- 2. Expansao dos protocolos (4 fraseados cada) ----
    _PREFIXO = "Explique o protocolo"        # ver build_dataset._exemplos_protocolos
    _TEMPLATES = [
        "Explique o protocolo {doc} - {titulo}.",
        "Qual a conduta recomendada pelo protocolo {doc}?",
        "Resuma os pontos principais do protocolo {titulo}.",
        "Quando devo aplicar o protocolo {doc}?",
    ]
    expandidos = []
    for x in _core:
        m = x.get("messages", [])
        if len(m) < 3 or not m[1]["content"].startswith(_PREFIXO):
            continue
        pergunta, resposta = m[1]["content"], m[2]["content"]
        mo = re.search(r"\[([A-Z]{2,}-?\d+)[^\]]*\]", resposta)
        if not mo:
            continue
        doc = mo.group(1)
        t = re.search(r" - (.+?)\.?\s*$", pergunta)
        titulo = t.group(1) if t else doc
        for frase in _TEMPLATES[: max(1, VARIACOES_PROTOCOLO)]:
            expandidos.append(_chat(SYSTEM_PT, frase.format(doc=doc, titulo=titulo),
                                    resposta, _rng.choice(FECHOS_PT)))
    print(f"expansao prot : {len(expandidos):>4}  ({VARIACOES_PROTOCOLO} fraseados)")

    # ---- 3. MedQuAD EN opcional, filtrado (default N_MEDQUAD_EN=0) ----
    medquad, descartados = [], 0
    if N_MEDQUAD_EN:
        mq = load_dataset("lavita/MedQuAD", split="train").shuffle(seed=SEED)
        for r in mq:
            p = str(r.get("question") or r.get("Question") or "").strip()
            a = str(r.get("answer") or r.get("Answer") or "").strip()
            if not p or not a:
                continue
            if not _medquad_ok(p, a):
                descartados += 1
                continue
            medquad.append(_chat(SYSTEM_EN, p, a, FECHO_EN))
            if len(medquad) >= N_MEDQUAD_EN:
                break
        print(f"MedQuAD EN    : {len(medquad):>4}  ({descartados} descartados pelo filtro)")

    exemplos = nucleo + expandidos + medquad
    _rng.shuffle(exemplos)
    n_pt = len(nucleo) + len(expandidos)
    frac_pt = n_pt / len(exemplos)
    print(f"\nTOTAL         : {len(exemplos):>4}   PT-BR {n_pt} ({frac_pt:.0%}) | EN {len(medquad)}")
    if frac_pt < FRACAO_PT_MIN:
        print(f"AVISO: fatia PT-BR ({frac_pt:.0%}) < {FRACAO_PT_MIN:.0%} - baixe N_MEDQUAD_EN.")

    Path(TRAIN_V3_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(TRAIN_V3_PATH, "w", encoding="utf-8") as fh:
        for ex in exemplos:
            fh.write(json.dumps(ex, ensure_ascii=False) + "\n")
    dataset = Dataset.from_list(exemplos)
    print(f"cache -> {TRAIN_V3_PATH}")


## 4. Train (QLoRA via Unsloth)

In [ ]:
import os
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTConfig, SFTTrainer

BASE_MODEL = 'unsloth/Llama-3.2-3B-Instruct'
# dataset v3 e pequeno (~116) -> 3 epocas; olhar a eval loss e cair p/ 2 se subir.
R, ALPHA, LR, EPOCHS, MAX_SEQ_LEN = 16, 32, 2e-4, 3, 2048

modelo, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True,
)
modelo = FastLanguageModel.get_peft_model(
    modelo, r=R, lora_alpha=ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42,
)

# `dataset` (train) vem da celula 3 (v3, ~116 ex. PT-BR). Esta versao do SFTTrainer
# nao aplica o chat template sozinho -> converter `messages` numa coluna `text`
# (template llama-3.1 cobre o Llama 3.2).
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

def _formatar(ex):
    return {'text': tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(_formatar, remove_columns=dataset.column_names)
val_ds = (load_dataset('json', data_files=VAL_PATH, split='train')
          .map(_formatar, remove_columns=['messages'])) if os.path.exists(VAL_PATH) else None
print(dataset[0]['text'][:600])
print(f'\ntrain: {len(dataset)}  |  val: {len(val_ds) if val_ds else 0}')

config = SFTConfig(
    output_dir=OUT_DIR, per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='linear', warmup_steps=10,
    optim='adamw_8bit', weight_decay=0.01, logging_steps=5,
    eval_strategy='epoch' if val_ds else 'no', save_strategy='epoch',
    max_seq_length=MAX_SEQ_LEN, dataset_text_field='text', seed=42, report_to='none',
)
trainer = SFTTrainer(
    model=modelo, args=config, train_dataset=dataset, eval_dataset=val_ds, tokenizer=tokenizer,
)
# so aprende com os turnos do assistant (mascara system/user da loss) - guia §4 celula 5
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>',
    response_part='<|start_header_id|>assistant<|end_header_id|>',
)
stats = trainer.train()
print(stats.metrics)
if val_ds:
    print('eval:', trainer.evaluate())

modelo.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f'Adaptadores salvos em {OUT_DIR}')


## 4b. Sanity check qualitativo (ANTES de exportar)

Gera 2-3 respostas com o modelo recem-treinado e confere: **cita `[PROT-...]`?**,
**termina sozinho (EOS)?**, **PT-BR?**. Se degenerar em loop ou nao citar ->
o problema e o dataset, nao os hiperparametros (guia §7 item 4). **So exportar o
GGUF se este check passar.**


In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(modelo)

_PERGUNTAS = [
    'Qual a conduta inicial na sepse no adulto?',
    'Existe protocolo institucional para hemorragia digestiva alta?',
    'Quando escalonar no manejo do delirium no idoso?',
]
for q in _PERGUNTAS:
    msgs = [{'role': 'system', 'content': SYSTEM_PT},
            {'role': 'user', 'content': q}]
    inputs = tokenizer.apply_chat_template(
        msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
    out = modelo.generate(input_ids=inputs, max_new_tokens=320, temperature=0.2,
                          do_sample=True, repetition_penalty=1.15,
                          eos_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    parou = out[0][-1].item() in (tokenizer.eos_token_id
                                  if isinstance(tokenizer.eos_token_id, list)
                                  else [tokenizer.eos_token_id])
    print('=' * 70)
    print('Q:', q)
    print('cita [PROT-...]:', bool(re.search(r'\[PROT-\d+', txt)), '| parou (EOS):', parou)
    print('-' * 70)
    print(txt)


## 5. Export (merge + GGUF Q4_K_M)

`save_pretrained_gguf` faz merge do LoRA + conversao + quantizacao numa chamada (compila o
llama.cpp na 1a vez, ~15-30 min na T4). Os intermediarios pesados ficam em `/content`; so o GGUF
final (~2 GB) e copiado para o Drive, entao um novo crash nao obriga a refazer tudo.

In [ ]:
import glob
import os
import shutil

from unsloth import FastLanguageModel

!df -h /content | tail -1        # ~16-20 GB de pico aqui durante o export
!free -g | awk 'NR==1||NR==2'    # RAM: o merge 16-bit e o passo que mais consome

GGUF_LOCAL = '/content/medassist-gguf'                       # Unsloth grava em <isto>_gguf/
GGUF_DRIVE = '/content/drive/MyDrive/medassist'             # so o .gguf final vai pra ca
QUANT = 'q4_k_m'

FastLanguageModel.for_inference(modelo)

# merge (16-bit) + convert HF->GGUF + quantiza Q4_K_M; baixa/compila o llama.cpp na 1a vez.
# maximum_memory_usage baixo -> menos risco de OOM na RAM (~12.7 GB no Colab free).
modelo.save_pretrained_gguf(
    GGUF_LOCAL, tokenizer, quantization_method=QUANT, maximum_memory_usage=0.6,
)

# Localiza o GGUF gerado (o nome/pasta varia por versao do Unsloth) e copia so ele
# (~2 GB) para o Drive -> sobrevive a um novo crash.
_ggufs = glob.glob('/content/**/*.gguf', recursive=True)
if not _ggufs:
    raise FileNotFoundError("Nenhum .gguf em /content - ver a saida do save_pretrained_gguf acima.")
_q4 = [f for f in _ggufs if QUANT in f.lower()]
_origem = max(_q4 or _ggufs, key=os.path.getsize)
os.makedirs(GGUF_DRIVE, exist_ok=True)
_destino = f'{GGUF_DRIVE}/medassist-q4_k_m.gguf'
shutil.copy(_origem, _destino)
print('GGUF gerado :', _origem, f'({os.path.getsize(_origem) / 1e9:.2f} GB)')
print('copiado p/  :', _destino)

# Libera espaco local depois da copia (descomente se o /content ficar apertado):
# shutil.rmtree(f'{GGUF_LOCAL}_gguf', ignore_errors=True)

## 5. Export (merge + GGUF)

In [ ]:
# O GGUF ja esta salvo no Drive pela celula 5 (MyDrive/medassist/medassist-q4_k_m.gguf).
# Opcoes para levar ate a VPS (em ./models/medassist-q4_k_m.gguf):
#   a) baixar direto do Google Drive pelo navegador (mais confiavel p/ ~2 GB);
#   b) subir a um repo privado no HF Hub e `hf download` na VPS (ver docs/finetuning.md §4/§8);
#   c) o download abaixo (pode falhar/reiniciar em arquivos grandes).
from google.colab import files

files.download('/content/drive/MyDrive/medassist/medassist-q4_k_m.gguf')

## 6. Download

In [ ]:
from google.colab import files
files.download('/content/medassist-q4_k_m.gguf')

# Ou copiar para o Drive e baixar depois com scp/rsync para a VPS em ./models/